# 🏆 Google Colab Stockfish 19 Dev/Master Analysis Server

This Jupyter Notebook powers the Stockfish 19 analysis engine for the **Chess.com-Style Game Review Platform**.
Follow the 5 steps below to launch the backend analysis server and link it to your Render web application.

---

## STEP 1 — Install Dependencies
Installs required build tools, system libraries, Python dependencies, and Cloudflare Tunnel CLI (`cloudflared`).

In [ ]:
# STEP 1 — Install System & Python Dependencies
!apt-get update -qq
!apt-get install -y -qq build-essential git python3 python3-pip curl wget lscpu

# Install Python packages required for the FastAPI Stockfish server
!pip install -q python-chess fastapi uvicorn pydantic requests httpx

# Install Cloudflare Tunnel CLI (cloudflared)
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

print("✅ STEP 1 Complete: All dependencies installed successfully.")

## STEP 2 — Detect CPU & Build Stockfish 19 Dev/Master
Automatically detects CPU architecture capability (AVX2 / BMI2 / x86-64-modern) and compiles official Stockfish 19 Dev/Master source code.

In [ ]:
# STEP 2 — Clone & Compile Stockfish 19 Dev/Master Engine
import os
import subprocess

print("🔍 Detecting CPU Capabilities...")
cpu_info = subprocess.getoutput("lscpu")
arch = "x86-64-modern"

if "avx512" in cpu_info.lower():
    arch = "x86-64-avx512"
elif "bmi2" in cpu_info.lower() or "avx2" in cpu_info.lower():
    arch = "x86-64-bmi2"
elif "popcnt" in cpu_info.lower():
    arch = "x86-64-modern"
else:
    arch = "x86-64"

print(f"⚡ Optimal CPU Architecture selected: {arch}")

# Clone Stockfish official repository if not present
if not os.path.exists("Stockfish"):
    print("📥 Cloning official Stockfish repository...")
    !git clone --depth 1 https://github.com/official-stockfish/Stockfish.git

print("🛠️ Compiling Stockfish 19 Dev/Master...")
%cd Stockfish/src
!make -j$(nproc) build ARCH={arch}

# Verify binary
sf_path = os.path.abspath("stockfish")
if os.path.exists(sf_path):
    print(f"✅ Binary created at: {sf_path}")
    os.environ["STOCKFISH_PATH"] = sf_path
    # Copy to /usr/local/bin so system finds it directly
    !cp stockfish /usr/local/bin/stockfish
    !chmod +x /usr/local/bin/stockfish
    %cd /content
    
    # Test stockfish with uci command
    p = subprocess.Popen(["/usr/local/bin/stockfish"], stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
    out, _ = p.communicate(input="uci\nquit\n", timeout=5)
    if "id name Stockfish" in out:
        first_line = [line for line in out.splitlines() if "id name Stockfish" in line][0]
        print(f"🎉 Verified Engine Output: {first_line}")
        print("✅ STEP 2 Complete: Stockfish 19 Dev/Master built and verified!")
    else:
        print("⚠️ Engine test output warning:", out)
else:
    raise RuntimeError("Stockfish compilation failed. Binary not found.")

## STEP 3 — Start FastAPI Server
Launches the FastAPI backend service in the background on `http://127.0.0.1:8000`.

In [ ]:
# STEP 3 — Clone application code & Start FastAPI Server in Background
import os
import time
import subprocess
import requests

# If running directly in Colab without repo clone:
if not os.path.exists("backend"):
    print("📥 Fetching backend application files from GitHub...")
    !git clone https://github.com/your-username/chess-game-review.git /content/app || true
    if os.path.exists("/content/app/backend"):
        %cd /content/app

print("🚀 Starting FastAPI server on http://127.0.0.1:8000...")
server_process = subprocess.Popen(
    ["python3", "-m", "uvicorn", "backend.app:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)
try:
    r = requests.get("http://127.0.0.1:8000/api/health", timeout=3)
    if r.status_code == 200:
        print("✅ FastAPI Server is running locally:", r.json())
        print("✅ STEP 3 Complete.")
    else:
        print("⚠️ Server returned unexpected status:", r.status_code)
except Exception as e:
    print("⚠️ Waiting for server to initialize...", e)

## STEP 4 — Launch Public Cloudflare Tunnel
Exposes your local FastAPI server publicly over HTTPS via Cloudflare Tunnel.

In [ ]:
# STEP 4 — Launch Cloudflare Tunnel and Extract HTTPS Public URL
import re
import time
import subprocess

print("🌐 Opening Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
start_time = time.time()

while time.time() - start_time < 30:
    line = tunnel_process.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    if "trycloudflare.com" in line:
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print("\n" + "=" * 50)
    print("🏆 CHESS ANALYSIS SERVER IS LIVE!")
    print("=" * 50)
    print(f"Local URL : http://127.0.0.1:8000")
    print(f"Public URL: {public_url}")
    print("=" * 50)
    print(f"👉 Copy '{public_url}' and paste it into Settings -> Colab Server URL in your Render App.")
    print("=" * 50 + "\n")
else:
    print("❌ Tunnel creation timed out. Please re-run STEP 4.")

## STEP 5 — Server Keep-Alive & Health Polling
Performs continuous health checks to prevent idle Colab disconnects.

> **Note:** Google Colab runtimes are temporary by design and max out after 12–24 hours or on inactivity. Keep this tab open while reviewing games.

In [ ]:
# STEP 5 — Health Check Polling & Runtime Keep-Alive
import time
import requests
from datetime import datetime

print("🔄 Starting keep-alive polling loop. Press Stop in Colab to terminate server.")
poll_count = 0

try:
    while True:
        time.sleep(30)
        poll_count += 1
        now = datetime.now().strftime("%H:%M:%S")
        try:
            res = requests.get("http://127.0.0.1:8000/api/health", timeout=2)
            if res.status_code == 200:
                print(f"[{now}] Heartbeat #{poll_count}: Engine OK (Stockfish 19 Running)")
            else:
                print(f"[{now}] Warning: Health check returned {res.status_code}")
        except Exception as err:
            print(f"[{now}] Connection check warning: {err}")
except KeyboardInterrupt:
    print("🛑 Server keep-alive stopped by user.")